# 01 · Data Acquisition — Besaya Basin (Los Corrales de Buelna)

**Author:** Salvador Navas  
**Basin:** Río Besaya — Cantabria (Spain)  
**Approx. area:** ~700 km²  

## Basin context

The Besaya flows northwest through industrial Cantabria, passing Los Corrales de Buelna 
and discharging into the Ría de San Martín at Torrelavega. The basin has an 
**Atlantic/Cantabrian climate**: high total annual precipitation (900–1,800 mm/yr), 
concentrated in autumn and winter, with summer minima. Flash floods from convective 
summer storms and slow frontal floods from long winter spells are both important 
design scenarios.

**Why two data types?**

| Data | Source | Use in the workflow |
|------|--------|---------------------|
| **Daily rainfall** — 13 AEMET gauges | AEMET Open Data API | Spatial interpolation → AMR per subbasin (NB02) |
| **Daily streamflow** — 2 SAIH gauges | CHC / SAIH Cantabria | HMS calibration target (NB05), event extraction (NB06) |

## Workflow
1. Path and parameter configuration
2. AEMET data download (API) — optional if local CSVs already exist
3. Load prepared rain gauge records (13 stations)
4. Load streamflow gauges (stations 1237-Torrelavega and 1937-Las Caldas)
5. Basic quality control
6. Visualisation: time series and station map
7. Export clean data for use in subsequent notebooks


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR  = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))
DATA_ROOT = DATA_DIR / 'pilot_cases' / 'los_corrales_buelna'
RAIN_DIR  = DATA_ROOT / 'gauges' / 'rainfall'
FLOW_DIR  = DATA_ROOT / 'gauges' / 'flow'
OUT_DIR   = DATA_ROOT / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('Archivos pluviometría:', len(list(RAIN_DIR.glob('*.csv'))))

---
## 1. Rain gauge station metadata

The metadata file `Datos_estaciones_X_Y_Pmean.csv` contains UTM 30N coordinates 
(columns `X`, `Y` in metres) and mean annual precipitation `Pmean` (mm/yr) for 
each of the 13 AEMET stations in the Besaya basin.

`Pmean` is used in two ways:
- As the **interpolation target** for building the mean annual precipitation map (NB02)
- As a **spatial consistency check** during QC — stations with `Pmean` very different 
  from their geographic neighbours may have moved or have systematic instrumental errors


In [2]:
stations_meta = pd.read_csv(RAIN_DIR / 'Datos_estaciones_X_Y_Pmean.csv')
stations_meta.columns = stations_meta.columns.str.strip()
print(f'{len(stations_meta)} estaciones — coordenadas UTM ETRS89 Huso 30N')
stations_meta

13 estaciones — coordenadas UTM ETRS89 Huso 30N


---
## 2. AEMET download (optional — requires API key)

The AEMET Open Data API provides free historical daily data for all Spanish gauging 
network stations. Registration is free at https://opendata.aemet.es.

If the prepared CSV files (`Pluvios_conjunto.csv`) are already available in the 
data directory, skip this step — the download is slow (~2 min for 13 stations 
× 50 years at daily resolution). Set `HYDRA_RUN_DOWNLOADS=1` to re-download.

Available variables in AEMET daily:
- `prec` — daily precipitation (mm, midnight-to-midnight)
- `tmin`, `tmax` — minimum and maximum temperature (°C)
- `racha`, `velmedia` — wind gust and mean speed (m/s)


In [3]:
# from pyhydra.data_sources.rainfall import download_aemet_daily_data
#
# API_KEY    = 'TU_API_KEY_AEMET'   # https://opendata.aemet.es/centrodedescargas/altaUsuario
# AEMET_DIR  = DATA_ROOT / 'aemet_raw'
# AEMET_DIR.mkdir(exist_ok=True)
#
# AEMET stations in the Besaya basin (8-digit codes)
# IDs aproximados: 1117B, 1120, 1121O, 1124, 1127, 1127U, 1135, 1136A, 1139D, 1140E, 1144, 1151, 1153E
#
# download_aemet_daily_data(
#     path_output=str(AEMET_DIR),
#     api_key=API_KEY,
#     start_date_str='1970-01-01',
#     end_date_str='2023-12-31',
# )

print('Datos pluviométricos ya disponibles en RAIN_DIR → se omite descarga AEMET')

Datos pluviométricos ya disponibles en RAIN_DIR → se omite descarga AEMET


---
## 3. Load prepared rain gauge data

`Pluvios_conjunto.csv` is a pre-assembled matrix (date × station) with daily 
precipitation in mm. The date format is `m/d/Y` (US locale from the original 
AEMET export) — `pd.read_csv(..., dayfirst=False)` handles this correctly.

All columns are renamed to station codes matching the metadata file so that 
later joins (NB02 spatial interpolation) work without manual mapping.


In [4]:
# Pluvios_conjunto.csv already contains all 13 stations with date index in m/d/Y format
RAIN_COMBINED = RAIN_DIR / 'Pluvios_conjunto.csv'

rain = pd.read_csv(RAIN_COMBINED, index_col=0, parse_dates=True,
                   dayfirst=False)
rain.index = pd.to_datetime(rain.index, format='%m/%d/%Y', errors='coerce')
rain = rain.dropna(how='all').clip(lower=0)
# Alinear nombres de columnas con los de stations_meta
rain.columns = [c.upper() for c in rain.columns]

print(f'Pluvios combinados: {rain.shape[0]} días × {rain.shape[1]} estaciones')
print(f'Período: {rain.index[0].date()} → {rain.index[-1].date()}')
print(f'Estaciones: {list(rain.columns)}')

# ── How to load from scratch with individual CSVs (Ano/Mes/Dia/mm) ───────────
# def _load_rain_csv(path):
#     df = pd.read_csv(path)
#     df['date'] = pd.to_datetime(df[['Ano','Mes','Dia']].rename(
#         columns={'Ano':'year','Mes':'month','Dia':'day'}))
#     return df.set_index('date')['mm'].rename(path.stem.upper()).clip(lower=0)
# rain = pd.concat([_load_rain_csv(f) for f in RAIN_DIR.glob('1*.csv')], axis=1)
# ──────────────────────────────────────────────────────────────────────────────

Pluvios combinados: 15433 días × 13 estaciones
Período: 1970-10-01 → 2012-12-31
Estaciones: ['1117B', '1120', '1121O', '1124', '1127', '1127U', '1135', '1136A', '1139D', '1140E', '1144', '1151', '1153E']


---
## 4. Streamflow gauge data

Two streamflow gauges bracket the main reach of the Besaya:
- **1237 Torrelavega** (outlet, ~700 km²): longest record; primary calibration target
- **1937 Las Caldas** (upper reach, ~200 km²): shorter record; useful for upstream validation

Data format: Excel files (`.xls`/`.xlsx`) from the CHC (Confederación Hidrográfica del Cantábrico) 
SAIH network. Each file contains daily mean discharge in m³/s, date column, 
and a quality flag (G = good, M = missing, E = estimated). Missing values coded as 
-999 are replaced by NaN.

> **Note on units:** SAIH files report discharge in m³/s. HEC-HMS can operate in 
> metric (m³/s) or imperial (ft³/s) — check the project units before writing `.dss`.


In [5]:
flow_files = list(FLOW_DIR.glob('*.xls')) + list(FLOW_DIR.glob('*.xlsx'))
print('Archivos de aforo:', [f.name for f in flow_files])

flow_series = {}
for fpath in flow_files:
    df = None
    try:
        # Find the header row that contains 'Caudal' — skip title/metadata rows
        for header_row in range(10):
            try:
                df_try = pd.read_excel(fpath, header=header_row)
                df_try.columns = [str(c).strip() for c in df_try.columns]
                if any('Caudal' in c or 'caudal' in c.lower() for c in df_try.columns):
                    df = df_try
                    break
            except Exception:
                continue

        if df is None:
            print(f'  ⚠  {fpath.name}: no se pudo leer o no contiene columna Caudal')
            continue

        q_col = next((c for c in df.columns if 'Caudal' in c or 'caudal' in c.lower()), None)
        yr_col = next((c for c in df.columns if 'ño' in c or c.lower() in ['ano', 'año']), None)

        # Unnamed: 6 contains clean full-date Timestamps in both file formats — prefer it
        if 'Unnamed: 6' in df.columns:
            df['date'] = pd.to_datetime(df['Unnamed: 6'], errors='coerce')
        elif yr_col and hasattr(df[yr_col].dropna().iloc[0], 'year'):
            df['date'] = pd.to_datetime(df[yr_col], errors='coerce')
        elif yr_col:
            mes_col = next((c for c in df.columns if c.lower() == 'mes'), None)
            dia_col = next((c for c in df.columns if c.lower() == 'dia'), None)
            if mes_col and dia_col:
                yrs = df[yr_col].astype(float).astype(int)
                yrs = yrs.apply(lambda y: y + 1900 if y < 100 else y)
                df['date'] = pd.to_datetime(
                    {'year': yrs, 'month': df[mes_col].astype(float).astype(int),
                     'day': df[dia_col].astype(float).astype(int)}, errors='coerce')
            else:
                print(f'  ⚠  {fpath.name}: columnas Mes/Dia no encontradas')
                continue
        else:
            print(f'  ⚠  {fpath.name}: estructura de fechas no reconocida')
            continue

        s = df.dropna(subset=['date']).set_index('date')[q_col].astype(float).clip(lower=0)
        s = s.groupby(s.index).first()
        station_id = fpath.stem.split('(')[0].strip()
        flow_series[station_id] = s.rename(station_id)
        print(f'  {fpath.name}: {len(s)} días ({s.index[0].date()} → {s.index[-1].date()})')

    except Exception as e:
        print(f'  ⚠  {fpath.name}: {e}')

flow = pd.concat(flow_series.values(), axis=1) if flow_series else pd.DataFrame()
print(f'\nDataFrame aforos: {flow.shape}')
if not flow.empty:
    print(flow.describe().round(1))
flow.head()

Archivos de aforo: ['1237 (Torrelavega)_actualizado.xls', '1937 (Las Caldas ).xls']
  1237 (Torrelavega)_actualizado.xls: 4340 días (2000-10-01 → 2012-08-18)
  1937 (Las Caldas ).xls: 10957 días (1970-10-01 → 2000-09-30)

DataFrame aforos: (15297, 2)
         1237     1937
count  2922.0  10957.0
mean     11.7     12.5
std      20.7     28.6
min       0.1      0.0
25%       2.8      2.1
50%       5.2      4.7
75%      11.9     12.6
max     285.1   1005.0


---
## 5. Quality control

Minimum QC checks before interpolation and modelling:

| Check | Threshold | Action |
|-------|-----------|--------|
| Missing data fraction | > 10% in any year → exclude from frequency analysis | Flag station |
| Negative values | Any `P < 0` | Replace with NaN |
| Daily precipitation > 200 mm | Suspicious for Cantabria | Verify against neighbours |
| Streamflow < 0 | Sensor/transcription error | Replace with NaN |
| Flat signal > 5 days | Frozen sensor or data repeating | Investigate manually |
| Streamflow / Area ratio | > 50 mm/day specific runoff | Check if an upstream dam released water |

The statistics printed here (`start`, `end`, `n_obs`, `missing_pct`) serve as a 
permanent record in the notebook output so future users can see the data quality 
at the time of the analysis.


In [6]:
print('=== Pluviometría ===')
qc_rain = pd.DataFrame({
    'inicio':   rain.apply(lambda s: s.first_valid_index()),
    'fin':      rain.apply(lambda s: s.last_valid_index()),
    'n_dias':   rain.count(),
    'nulos_%':  (rain.isna().mean() * 100).round(1),
    'max_mm':   rain.max().round(1),
    'mean_mm':  rain.mean().round(1),
})
print(qc_rain.to_string())

# Remove negative values or values above 500 mm/day (errors)
rain = rain.clip(lower=0, upper=500)

# Common period with data in at least 80% of stations
threshold = int(0.8 * rain.shape[1])
rain_valid = rain[rain.notna().sum(axis=1) >= threshold]
print(f'\nPeriodo con ≥80% estaciones activas: {rain_valid.index[0].date()} → {rain_valid.index[-1].date()}')

=== Pluviometría ===
          inicio        fin  n_dias  nulos_%  max_mm  mean_mm
1117B 1970-10-01 2012-12-31   15433      0.0   153.3      4.8
1120  1970-10-01 2012-12-31   15433      0.0   141.7      4.1
1121O 1970-10-01 2012-12-31   15433      0.0    89.7      3.6
1124  1970-10-01 2012-12-31   15433      0.0   115.8      4.3
1127  1970-10-01 2012-12-31   15433      0.0    98.1      4.3
1127U 1970-10-01 2012-12-31   15433      0.0   104.2      4.1
1135  1970-10-01 2012-12-31   15433      0.0   102.4      3.5
1136A 1970-10-01 2012-12-31   15433      0.0   100.5      3.6
1139D 1970-10-01 2012-12-31   15433      0.0    95.2      3.8
1140E 1970-10-01 2012-12-31   15433      0.0    82.8      3.0
1144  1970-10-01 2012-12-31   15433      0.0    89.5      3.1
1151  1970-10-01 2012-12-31   15433      0.0   122.2      3.5
1153E 1970-10-01 2012-12-31   15433      0.0    79.3      3.5

Periodo con ≥80% estaciones activas: 1970-10-01 → 2012-12-31


---
## 6. Visualisation

Two panels:
1. **Precipitation time series** — all 13 stations overlaid; vertical alignment of peaks 
   across stations confirms spatially coherent events (frontal rain). Isolated peaks at 
   one station only may indicate convective cells or gauge errors.
2. **Streamflow and station map** — the spatial distribution of gauges relative to the 
   basin outline reveals potential coverage gaps (unguarded western slopes near Reinosa 
   can produce fast-responding sub-basins with poor rain gauge coverage).

Inspect the station map to identify:
- Stations outside the basin polygon (will bias Kriging if included)
- Elevation gradient (Pmean should increase northward toward the coast and with elevation)


In [7]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# Series de lluvia
ax = axes[0]
for col in rain.columns:
    ax.plot(rain.index, rain[col], lw=0.4, alpha=0.6, label=col)
ax.set_ylabel('Precipitación (mm/día)')
ax.set_title('Pluviómetros — cuenca del Besaya')
ax.legend(ncol=4, fontsize=7, loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Series de caudal
ax = axes[1]
for col in flow.columns:
    ax.plot(flow.index, flow[col], lw=0.7, label=col)
ax.set_ylabel('Caudal (m³/s)' if not flow.empty else 'Sin datos de aforo')
ax.set_title('Aforos — cuenca del Besaya')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(OUT_DIR / 'series_temporales.png', dpi=150)
plt.show()

In [8]:
# Rain gauge station map with basin outline
fig, ax = plt.subplots(figsize=(8, 7))

# Cargar shapefile de la cuenca (Subbasin288.shp copiado a gauges/)
SHP_BASIN = DATA_ROOT / 'gauges' / 'Subbasin288.shp'
try:
    import geopandas as gpd
    if SHP_BASIN.exists():
        gdf = gpd.read_file(str(SHP_BASIN))
        gdf_utm = gdf.to_crs(epsg=25830) if gdf.crs else gdf
        gdf_utm.plot(ax=ax, color='lightblue', edgecolor='steelblue', alpha=0.4, linewidth=1.2)
    else:
        # Fallback: buscar cualquier shapefile en el modelo HMS
        shp_files = list((DATA_ROOT / 'models' / 'hec_hms').rglob('*.shp'))
        if shp_files:
            gdf = gpd.read_file(str(shp_files[0]))
            gdf_utm = gdf.to_crs(epsg=25830) if gdf.crs else gdf
            gdf_utm.plot(ax=ax, color='lightblue', edgecolor='gray', alpha=0.4)
except Exception as e:
    print(f'Shapefile no cargado: {e}')

sc = ax.scatter(stations_meta['X'], stations_meta['Y'],
                c=stations_meta['Pmean'], cmap='Blues', s=80, zorder=5, edgecolors='k')
for _, row in stations_meta.iterrows():
    ax.annotate(row['ID'], (row['X'], row['Y']), textcoords='offset points',
                xytext=(5, 3), fontsize=8)

# Aforos
if not flow.empty and 'flow_meta' not in dir():
    # Centroides aproximados de estaciones de aforo (Torrelavega, Las Caldas)
    aforo_coords = {'1237': (425823, 4778218), '1937': (421500, 4784000)}
    for sid, (x, y) in aforo_coords.items():
        ax.scatter(x, y, marker='^', s=120, color='tomato', zorder=6, edgecolors='k')
        ax.annotate(f'Aforo {sid}', (x, y), textcoords='offset points',
                    xytext=(5, 3), fontsize=8, color='tomato')

plt.colorbar(sc, ax=ax, label='Pmean (mm/año)')
ax.set_xlabel('X UTM (m)')
ax.set_ylabel('Y UTM (m)')
ax.set_title('Estaciones pluviométricas — Cuenca del Besaya')
plt.tight_layout()
plt.savefig(OUT_DIR / 'mapa_estaciones.png', dpi=150)
plt.show()

---
## 7. Export clean data

Three files are exported to `PROC_DIR` for downstream notebooks:

| File | Content | Used in |
|------|---------|---------|
| `rain_daily.csv` | Daily precipitation matrix (mm), date-indexed | NB02 (interpolation) |
| `flow_daily.csv` | Daily discharge matrix (m³/s) | NB05 (calibration), NB06 (events) |
| `stations_meta.csv` | Coordinates + Pmean per station | NB02 (IDW/Kriging fit) |


In [9]:
rain.to_csv(OUT_DIR / 'rain_daily.csv')
flow.to_csv(OUT_DIR / 'flow_daily.csv')
stations_meta.to_csv(OUT_DIR / 'stations_meta.csv', index=False)

print('Exportados:')
for f in OUT_DIR.iterdir():
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

Exportados:
  hidrogramas_metHab_Tyear.png  (83.7 KB)
  cc_hms_results  (2.1 KB)
  hms_continuous  (0.2 KB)
  copula_scatter.png  (102.5 KB)
  hydrographs  (13.3 KB)
  cc_scenarios  (0.7 KB)
  kriging_pmean.png  (57.6 KB)
  seasonality_maxima.png  (35.1 KB)
  extreme_value_params.json  (0.4 KB)
  eventos_sinteticos.csv  (4160.1 KB)
  flood_maps  (0.2 KB)
  idf_curves.png  (258.2 KB)
  ras_results  (0.2 KB)
  pma_idw_daily.csv  (12767.8 KB)
  gev_diagnostic.png  (187.0 KB)
  copula_marginals.png  (81.0 KB)
  kmeans_hydro_types.png  (550.6 KB)
  eventos_maxdiss.csv  (13.2 KB)
  scatter_synthetic_vs_real.png  (231.9 KB)
  kmeans_clusters.png  (306.9 KB)
  stations_meta.csv  (0.7 KB)
  matriz_sintetica_maxdiss.csv  (149.8 KB)
  serie_caudal.png  (80.8 KB)
  mapa_estaciones.png  (159.1 KB)
  pma_subbasins.png  (151.9 KB)
  eventos_observados.csv  (9.0 KB)
  loocv_comparison.png  (92.6 KB)
  cc_eventos  (0.3 KB)
  maxdiss_coverage.png  (235.6 KB)
  hms_events  (0.3 KB)
  reconstruction_weigh